In [2]:
from pathlib import Path
import shutil
import subprocess
from pprint import pprint

ROOT = Path("/home/debi/jaime/repos/MR-EyeTrack")
STUDY = ROOT / "data" / "study"

# Ensure dcm2niix is available in PATH
dcm2niix_path = shutil.which("dcm2niix")
if dcm2niix_path is None:
    raise RuntimeError("dcm2niix not found in PATH. Install it first (e.g., apt install dcm2niix).")

print(f"Using dcm2niix: {dcm2niix_path}")

results = []
for n in range(1, 16):
    sid = f"sub-{n:03d}"
    primary = STUDY / sid / "dicom" / "csTFL_mp-rage_1mm-iso_CP_acc4.6_5_MR"
    fallback = STUDY / sid / "dicom"
    dcm_name = f"{sid}.dcm"

    if (primary / dcm_name).is_file():
        folder = primary
    elif (fallback / dcm_name).is_file():
        folder = fallback
    else:
        results.append({
            "subject": sid,
            "status": "skipped (no matching DICOM file)",
            "folder": None,
            "outputs": []
        })
        continue

    # Remove prior conversion outputs for this subject before re-running conversion.
    for p in list(folder.glob(f"{sid}*.nii")) + list(folder.glob(f"{sid}*.nii.gz")) + list(folder.glob(f"{sid}*.json")):
        p.unlink()

    # Use fixed output base name (sub-XXX) without series suffixes like _5.
    cmd = [
        "dcm2niix",
        "-z", "y",
        "-f", sid,
        "-o", str(folder),
        str(folder),
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    outputs = sorted([p.name for p in folder.glob(f"{sid}*.nii*")]) + sorted([p.name for p in folder.glob(f"{sid}*.json")])

    results.append({
        "subject": sid,
        "status": "converted" if proc.returncode == 0 else f"error ({proc.returncode})",
        "folder": str(folder),
        "outputs": outputs,
        "stderr_tail": "\n".join(proc.stderr.splitlines()[-5:]) if proc.stderr else ""
    })

pprint(results)

Using dcm2niix: /home/debi/miniconda3/envs/mreyetrack/bin/dcm2niix


[{'folder': '/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-001/dicom/csTFL_mp-rage_1mm-iso_CP_acc4.6_5_MR',
  'outputs': ['sub-001.nii.gz', 'sub-001.json'],
  'status': 'converted',
  'stderr_tail': '',
  'subject': 'sub-001'},
 {'folder': '/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-002/dicom/csTFL_mp-rage_1mm-iso_CP_acc4.6_5_MR',
  'outputs': ['sub-002.nii.gz', 'sub-002.json'],
  'status': 'converted',
  'stderr_tail': '',
  'subject': 'sub-002'},
 {'folder': '/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-003/dicom/csTFL_mp-rage_1mm-iso_CP_acc4.6_5_MR',
  'outputs': ['sub-003.nii.gz', 'sub-003.json'],
  'status': 'converted',
  'stderr_tail': '',
  'subject': 'sub-003'},
 {'folder': '/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-004/dicom/csTFL_mp-rage_1mm-iso_CP_acc4.6_5_MR',
  'outputs': ['sub-004.nii.gz', 'sub-004.json'],
  'status': 'converted',
  'stderr_tail': '',
  'subject': 'sub-004'},
 {'folder': '/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-005/dicom